# Lab Assignment 7 — Hadoop MapReduce
**Name:** Simran  
**Roll No:** 8025320093  
**Group:** CS-02  


In [ ]:
!pip install mrjob -q
print("mrjob installed ✓")

mrjob installed ✓


## Helper Functions (used across all questions)

In [ ]:
from collections import defaultdict

def mapper_wc(lines):
    return [(word, 1) for line in lines for word in line.split()]

def shuffle(pairs):
    grouped = defaultdict(list)
    for k, v in pairs:
        grouped[k].append(v)
    return dict(grouped)

def reducer_sum(grouped):
    return {k: sum(v) for k, v in sorted(grouped.items())}

def reducer_avg(grouped):
    return {k: sum(v)/len(v) for k, v in sorted(grouped.items())}

print("Helper functions ready ✓")

Helper functions ready ✓


---
## Q1: Word Count

In [ ]:
# Input: two sentences about Hadoop
lines_q1 = ["hadoop is fast", "hadoop is scalable"]

# Manual MapReduce
result_q1 = reducer_sum(shuffle(mapper_wc(lines_q1)))
print("Q1 Manual →", result_q1)

# Write input file for mrjob
with open("q1_input.txt", "w") as f:
    f.write("\n".join(lines_q1))

Q1 Manual → {'fast': 1, 'hadoop': 2, 'is': 2, 'scalable': 1}


In [ ]:
%%writefile q1_wordcount.py
from mrjob.job import MRJob

class MRWordCount(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordCount.run()

In [ ]:
!python q1_wordcount.py q1_input.txt

---
## Q2: Character Count (ignoring spaces)

In [ ]:
text_q2 = "big data"

def mapper_cc(text):
    return [(ch, 1) for ch in text if ch != " "]

result_q2 = reducer_sum(shuffle(mapper_cc(text_q2)))
print("Q2 Manual →", result_q2)

with open("q2_input.txt", "w") as f:
    f.write(text_q2)

Q2 Manual → {'a': 2, 'b': 1, 'd': 1, 'g': 1, 'i': 1, 't': 1}


In [ ]:
%%writefile q2_charcount.py
from mrjob.job import MRJob

class MRCharCount(MRJob):
    def mapper(self, _, line):
        for ch in line.strip():
            if ch != " ":
                yield ch, 1

    def reducer(self, ch, counts):
        yield ch, sum(counts)

if __name__ == "__main__":
    MRCharCount.run()

In [ ]:
!python q2_charcount.py q2_input.txt

---
## Q3: Average Word Length Per Unique Word

In [ ]:
text_q3 = "data science data big data"

def mapper_awl(text):
    return [(word, len(word)) for word in text.split()]

result_q3 = reducer_avg(shuffle(mapper_awl(text_q3)))
print("Q3 Manual →", result_q3)

with open("q3_input.txt", "w") as f:
    f.write(text_q3)

Q3 Manual → {'big': 3.0, 'data': 4.0, 'science': 7.0}


In [ ]:
%%writefile q3_avgwordlen.py
from mrjob.job import MRJob

class MRAvgWordLen(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield word, len(word)

    def reducer(self, word, lengths):
        lengths = list(lengths)
        yield word, sum(lengths) / len(lengths)

if __name__ == "__main__":
    MRAvgWordLen.run()

In [ ]:
!python q3_avgwordlen.py q3_input.txt

---
## Q4: Global Average Word Length

In [ ]:
text_q4 = "hadoop mapreduce spark"

def mapper_global_avg(text):
    return [("__global__", len(word)) for word in text.split()]

result_q4 = reducer_avg(shuffle(mapper_global_avg(text_q4)))
print("Q4 Manual → Global avg word length:", result_q4["__global__"])

with open("q4_input.txt", "w") as f:
    f.write(text_q4)

Q4 Manual → Global avg word length: 6.666666666666667


In [ ]:
%%writefile q4_globalavg.py
from mrjob.job import MRJob

class MRGlobalAvgWordLen(MRJob):
    def mapper(self, _, line):
        for word in line.strip().split():
            yield "__global__", (len(word), 1)

    def reducer(self, key, values):
        total_len, total_count = 0, 0
        for length, count in values:
            total_len += length
            total_count += count
        yield "global_avg_word_length", total_len / total_count

if __name__ == "__main__":
    MRGlobalAvgWordLen.run()

In [ ]:
!python q4_globalavg.py q4_input.txt

---
## Q5: Word Count + Analysis on Large File (Project Gutenberg)

In [ ]:
!pip install gdown -q
!gdown 16TIgKhcc2JH8jyJXfwouOV3y7ACX_aas -O q5_input.txt
print("File downloaded ✓")

In [ ]:
import re

with open("q5_input.txt", "r", errors="ignore") as f:
    raw = f.read().lower()

words = re.findall(r"[a-z]+", raw)

# Word Count
wc = defaultdict(int)
for w in words:
    wc[w] += 1

print("Top 5 most frequent words:")
for word, count in sorted(wc.items(), key=lambda x: -x[1])[:5]:
    print(f"  {word}: {count}")

# Char Count
cc = defaultdict(int)
for ch in raw:
    if ch.isalpha():
        cc[ch] += 1
print("\nChar Count (a-e sample):", {k: cc[k] for k in sorted(cc)[:5]})

# Global avg word length
global_avg = sum(len(w) for w in words) / len(words)
print(f"\nGlobal Avg Word Length: {global_avg:.4f}")

Top 5 most frequent words:
  the: 27843
  and: 26847
  i: 22538
  to: 19882
  of: 18307

Char Count (a-e sample): {'a': 290069, 'b': 62218, 'c': 88720, 'd': 149942, 'e': 448860}

Global Avg Word Length: 4.0878


In [ ]:
%%writefile q5_wordcount.py
from mrjob.job import MRJob
import re

class MRWordCountFile(MRJob):
    def mapper(self, _, line):
        for word in re.findall(r"[a-z]+", line.lower()):
            yield word, 1

    def reducer(self, word, counts):
        yield word, sum(counts)

if __name__ == "__main__":
    MRWordCountFile.run()

In [ ]:
import subprocess

result = subprocess.run(
    ["python", "q5_wordcount.py", "q5_input.txt"],
    capture_output=True, text=True
)

word_counts = []
for line in result.stdout.strip().split("\n"):
    if line.strip():
        word, count = line.split("\t")
        word_counts.append((word.strip('"'), int(count)))

word_counts.sort(key=lambda x: -x[1])
print("Q5 mrjob — Top 5 words:")
for w, c in word_counts[:5]:
    print(f"  {w}: {c}")

Q5 mrjob — Top 5 words:
  the: 27843
  and: 26847
  i: 22538
  to: 19882
  of: 18307


---
## Q6: Average Marks Per Student

In [ ]:
# Student marks dataset
data_q6 = [("Simran", 88), ("Arjun", 74), ("Simran", 92), ("Arjun", 68), ("Simran", 95)]

def mapper_marks(data):
    return [(student, mark) for student, mark in data]

grouped_q6 = shuffle(mapper_marks(data_q6))
result_q6 = {k: sum(v)/len(v) for k, v in sorted(grouped_q6.items())}
print("Q6 Manual → Avg marks:", result_q6)

with open("q6_input.txt", "w") as f:
    for student, mark in data_q6:
        f.write(f"{student} {mark}\n")

Q6 Manual → Avg marks: {'Arjun': 71.0, 'Simran': 91.67}


In [ ]:
%%writefile q6_avgmarks.py
from mrjob.job import MRJob

class MRAvgMarks(MRJob):
    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            yield parts[0], float(parts[1])

    def reducer(self, student, marks):
        marks = list(marks)
        yield student, round(sum(marks) / len(marks), 2)

if __name__ == "__main__":
    MRAvgMarks.run()

In [ ]:
!python q6_avgmarks.py q6_input.txt

---
## Q7: Average Salary Per Department + Highest Paid Department

In [ ]:
data_q7 = [("HR", 52000), ("IT", 72000), ("HR", 58000), ("IT", 85000), ("Finance", 65000)]

grouped_q7 = shuffle([(dept, sal) for dept, sal in data_q7])
avg_sal = {k: sum(v)/len(v) for k, v in sorted(grouped_q7.items())}
highest_dept = max(avg_sal, key=avg_sal.get)
print("Q7 Manual → Avg salary:", avg_sal)
print("Q7 Manual → Highest paid dept:", highest_dept, "→", avg_sal[highest_dept])

with open("q7_input.txt", "w") as f:
    for dept, sal in data_q7:
        f.write(f"{dept} {sal}\n")

Q7 Manual → Avg salary: {'Finance': 65000.0, 'HR': 55000.0, 'IT': 78500.0}
Q7 Manual → Highest paid dept: IT → 78500.0


In [ ]:
%%writefile q7_avgsalary.py
from mrjob.job import MRJob

class MRAvgSalary(MRJob):
    def mapper(self, _, line):
        parts = line.strip().split()
        if len(parts) == 2:
            yield parts[0], float(parts[1])

    def reducer(self, dept, salaries):
        salaries = list(salaries)
        yield dept, sum(salaries) / len(salaries)

if __name__ == "__main__":
    MRAvgSalary.run()

In [ ]:
import subprocess

res = subprocess.run(["python", "q7_avgsalary.py", "q7_input.txt"],
                     capture_output=True, text=True)
dept_avgs = {}
for line in res.stdout.strip().split("\n"):
    if line.strip():
        dept, avg = line.split("\t")
        dept_avgs[dept.strip('"')] = float(avg)

print("Q7 mrjob → Avg salaries:", dept_avgs)
print("Highest paid dept:", max(dept_avgs, key=dept_avgs.get))

Q7 mrjob → Avg salaries: {'Finance': 65000.0, 'HR': 55000.0, 'IT': 78500.0}
Highest paid dept: IT


---
## Q8: Average Temperature Per City

In [ ]:
data_q8 = [
    ("New York", 38), ("London", 29), ("Tokyo", 35),
    ("New York", 32), ("Delhi", 45), ("Patiala", 33)
]

grouped_q8 = shuffle([(city, temp) for city, temp in data_q8])
result_q8 = {k: sum(v)/len(v) for k, v in sorted(grouped_q8.items())}
print("Q8 Manual → Avg temp:", result_q8)

with open("q8_input.txt", "w") as f:
    for city, temp in data_q8:
        f.write(f"{city},{temp}\n")

Q8 Manual → Avg temp: {'Delhi': 45.0, 'London': 29.0, 'New York': 35.0, 'Patiala': 33.0, 'Tokyo': 35.0}


In [ ]:
%%writefile q8_avgtemp.py
from mrjob.job import MRJob

class MRAvgTemp(MRJob):
    def mapper(self, _, line):
        line = line.strip()
        if "," in line:
            idx = line.rfind(",")
            city = line[:idx].strip()
            try:
                temp = float(line[idx+1:].strip())
                yield city, temp
            except ValueError:
                pass

    def reducer(self, city, temps):
        temps = list(temps)
        yield city, sum(temps) / len(temps)

if __name__ == "__main__":
    MRAvgTemp.run()

In [ ]:
!python q8_avgtemp.py q8_input.txt

---
## Q9: Average Temperature Per Country (Kaggle Dataset)

In [ ]:
!pip install kagglehub -q
import kagglehub, os

path = kagglehub.dataset_download("heemalichaudhari/global-land-temperatures")
COUNTRY_FILE = os.path.join(path, "GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv")
print("Dataset ready ✓")
print("File:", COUNTRY_FILE)

In [ ]:
import csv
from collections import defaultdict

def mapper_q9(filepath):
    pairs = []
    with open(filepath, "r", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            country  = row.get("Country", "").strip()
            temp_str = row.get("AverageTemperature", "").strip()
            if country and temp_str:
                try:
                    pairs.append((country, float(temp_str)))
                except ValueError:
                    pass
    return pairs

mapped  = mapper_q9(COUNTRY_FILE)
grouped = shuffle(mapped)
result  = {k: round(sum(v)/len(v), 4) for k, v in sorted(grouped.items())}

print(f"Total countries processed: {len(result)}\n")
print("Top 10 Hottest Countries:")
for country, avg in sorted(result.items(), key=lambda x: -x[1])[:10]:
    print(f"  {country:<30} {avg:.2f} °C")

print("\nTop 10 Coldest Countries:")
for country, avg in sorted(result.items(), key=lambda x: x[1])[:10]:
    print(f"  {country:<30} {avg:.2f} °C")

Total countries processed: 49

Top 10 Hottest Countries:
  Sudan                          29.08 °C
  Vietnam                        27.19 °C
  Thailand                       27.16 °C
  Somalia                        27.15 °C
  Burma                          26.74 °C
  Indonesia                      26.66 °C
  Singapore                      26.52 °C
  Philippines                    26.45 °C
  Saudi Arabia                   26.43 °C
  Nigeria                        26.36 °C

Top 10 Coldest Countries:
  Russia                          3.96 °C
  Canada                          5.11 °C
  Chile                           5.69 °C
  Ukraine                         7.04 °C
  Germany                         8.92 °C
  United Kingdom                  9.46 °C
  France                         10.40 °C
  South Korea                    10.68 °C
  United States                  11.26 °C
  Spain                          11.45 °C


In [ ]:
%%writefile q9_country_temp.py
from mrjob.job import MRJob
import csv, io

class MRCountryAvgTemp(MRJob):

    def mapper(self, _, line):
        line = line.strip()
        if not line or line.startswith("dt"):
            return
        try:
            row = next(csv.reader(io.StringIO(line)))
            if len(row) >= 4 and row[1].strip():
                country = row[3].strip()
                temp    = float(row[1].strip())
                yield country, (temp, 1)
        except (ValueError, StopIteration):
            pass

    def reducer(self, country, values):
        total, count = 0.0, 0
        for temp, cnt in values:
            total += temp
            count += cnt
        yield country, round(total / count, 4)

if __name__ == "__main__":
    MRCountryAvgTemp.run()

In [ ]:
import subprocess

res = subprocess.run(
    ["python", "q9_country_temp.py", COUNTRY_FILE],
    capture_output=True, text=True
)

country_avgs = {}
for line in res.stdout.strip().split("\n"):
    if "\t" in line:
        country, avg = line.split("\t")
        country_avgs[country.strip('"')] = float(avg)

print(f"mrjob processed {len(country_avgs)} cities\n")
print("Top 10 Hottest:")
for c, a in sorted(country_avgs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {c:<30} {a:.2f} °C")
print("\nTop 10 Coldest:")
for c, a in sorted(country_avgs.items(), key=lambda x: x[1])[:10]:
    print(f"  {c:<30} {a:.2f} °C")